In [1]:
# import requests
import pandas as pd
import numpy as np
from scipy.stats import entropy
import scipy.stats as stats

# import matplotlib.pyplot as plt
# import seaborn as sns
from statsmodels.tsa.stattools import adfuller  
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import root_mean_squared_error , mean_absolute_error , mean_absolute_percentage_error          
from sklearn.model_selection import TimeSeriesSplit


import itertools

import time
# import yaml
pd.options.display.float_format = '{:,.2f}'.format
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

## 1. Data Preparation

In [4]:
df = pd.read_csv("italy_unemployment_rate_PREDICTION_DATASET.csv")


In [5]:
df

,month,employment_rate_diff,industry_production_prices_diff,companies_trust,rolling_mean_6m,rolling_mean_12m,PCA_labour_market_1,PCA_labour_market_2,PCA_price_energy_1,PCA_price_energy_2,WorkingDays
0,2.00,1.12,0.00,87.30,0.50,0.07,1.01,0.01,0.06,0.13,NaN
1,3.00,0.17,-0.20,89.20,0.15,0.04,1.32,0.01,0.04,0.10,NaN
2,4.00,-1.24,-0.20,88.30,-0.00,0.03,-2.75,0.00,0.03,0.08,NaN
3,5.00,2.21,-0.10,87.60,-0.07,0.04,1.92,0.01,0.05,0.07,NaN
4,6.00,-0.40,0.20,89.30,-0.16,-0.03,-1.47,-0.01,0.04,0.07,NaN
...,...,...,...,...,...,...,...,...,...,...,...
151,NaN,0.04,0.13,96.27,0.02,NaN,NaN,NaN,NaN,-0.01,22.00
152,NaN,0.04,0.49,96.15,-0.02,NaN,NaN,NaN,NaN,0.01,21.00
153,NaN,0.04,0.04,95.81,-0.02,NaN,NaN,NaN,NaN,0.00,22.00
154,NaN,0.04,0.95,95.33,-0.07,NaN,NaN,NaN,NaN,0.00,23.00


In [14]:
df['date'] = pd.to_datetime(df['Unnamed: 0'])
X_columns = df.set_index('date')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

In [15]:
df.columns

Index(['Unnamed: 0', 'WorkingDays', 'fixed_term', 'employment_rate',
       'youth_employment_rate', 'industry_production_prices', 'diesel_price',
       'TotSales'],
      dtype='object')

In [16]:
df.drop('Unnamed: 0',axis = 1 , inplace = True)

In [17]:
df[:-12]

,WorkingDays,fixed_term,employment_rate,youth_employment_rate,industry_production_prices,diesel_price,TotSales
date,,,,,,,
2014-02-01,20,"2,116,627.00",38.64,54.49,93.00,1.64,"78,038,658.00"
2014-03-01,21,"2,188,435.00",38.81,55.42,92.80,1.63,"84,094,082.00"
2014-04-01,20,"2,215,274.00",37.57,54.69,92.60,1.63,"86,341,669.00"
2014-05-01,21,"2,255,743.00",39.78,55.44,92.50,1.63,"90,493,760.00"
2014-06-01,20,"2,436,989.00",39.38,55.90,92.70,1.63,"90,851,778.00"
...,...,...,...,...,...,...,...
2025-09-01,21,"2,501,210.00",44.11,62.55,123.10,1.64,"207,871,871.00"
2025-10-01,23,"2,532,114.00",43.28,62.78,122.80,1.62,"220,481,612.00"
2025-11-01,22,"2,378,014.00",43.00,62.13,124.00,1.68,"219,885,862.00"


In [18]:
df_sales = df['TotSales'].copy()
df.drop('TotSales', axis = 1, inplace = True)

In [20]:
df_sales.dropna(axis= 0, inplace =True)
df_sales

date
2014-02-01    78,038,658.00
2014-03-01    84,094,082.00
2014-04-01    86,341,669.00
2014-05-01    90,493,760.00
2014-06-01    90,851,778.00
                  ...      
2025-09-01   207,871,871.00
2025-10-01   220,481,612.00
2025-11-01   219,885,862.00
2025-12-01   222,564,373.00
2026-01-01   202,235,203.00
Name: TotSales, Length: 144, dtype: float64

In [24]:
numeric_cols = ['WorkingDays', 'fixed_term', 'employment_rate', 'youth_employment_rate', 
                'industry_production_prices', 'diesel_price']

stationary_cols = []
non_stationary_cols = []

for col in numeric_cols:
    print('---------------------------------------------')
    print(col)
    print('---------------------------------------------')

    test_staticist = adfuller(df[col])[0]
    adf_criticatical_value_5perc = adfuller(df[col])[4]['5%']
    if test_staticist > adf_criticatical_value_5perc:
        non_stationary_cols.append(col)
        print(f'{round(test_staticist,2)} > {round(adf_criticatical_value_5perc ,2)} ==> The time series is NON stationary')
        df[f'{col}_diff'] = df[col] - df[col].shift( 1) # qui facciamo subito la differnziazione

    else:
        stationary_cols.append(col)
        print(f'{round(test_staticist,2)} < {round(adf_criticatical_value_5perc ,2)} ==> The time series is STATIONARY')

---------------------------------------------
WorkingDays
---------------------------------------------
-1.63 > -2.88 ==> The time series is NON stationary
---------------------------------------------
fixed_term
---------------------------------------------
-2.62 > -2.88 ==> The time series is NON stationary
---------------------------------------------
employment_rate
---------------------------------------------
-2.05 > -2.88 ==> The time series is NON stationary
---------------------------------------------
youth_employment_rate
---------------------------------------------
-2.17 > -2.88 ==> The time series is NON stationary
---------------------------------------------
industry_production_prices
---------------------------------------------
-1.69 > -2.88 ==> The time series is NON stationary
---------------------------------------------
diesel_price
---------------------------------------------
-2.0 > -2.88 ==> The time series is NON stationary


In [25]:
df

,WorkingDays,fixed_term,employment_rate,youth_employment_rate,industry_production_prices,diesel_price,WorkingDays_diff,fixed_term_diff,employment_rate_diff,youth_employment_rate_diff,industry_production_prices_diff,diesel_price_diff
date,,,,,,,,,,,,
2014-02-01,20,"2,116,627.00",38.64,54.49,93.00,1.64,NaN,NaN,NaN,NaN,NaN,NaN
2014-03-01,21,"2,188,435.00",38.81,55.42,92.80,1.63,1.00,"71,808.00",0.17,0.93,-0.20,-0.01
2014-04-01,20,"2,215,274.00",37.57,54.69,92.60,1.63,-1.00,"26,839.00",-1.24,-0.73,-0.20,-0.00
2014-05-01,21,"2,255,743.00",39.78,55.44,92.50,1.63,1.00,"40,469.00",2.21,0.75,-0.10,0.00
2014-06-01,20,"2,436,989.00",39.38,55.90,92.70,1.63,-1.00,"181,246.00",-0.40,0.46,0.20,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-01,22,"2,714,840.53",42.35,59.79,90.85,1.62,0.00,0.00,0.36,-0.30,-4.37,0.00
2026-10-01,21,"2,714,840.53",42.14,59.57,87.28,1.58,-1.00,0.00,-0.21,-0.22,-3.57,-0.04
2026-11-01,22,"2,714,840.53",42.01,59.13,86.18,1.57,1.00,0.00,-0.13,-0.43,-1.10,-0.01


In [20]:
X_train = df.iloc[:-12].drop('TotSales', axis = 1)
X_test = df.iloc[-12:].drop('TotSales', axis = 1)

y_train = df.iloc[:-12]['TotSales']
y_test = df.iloc[-12:]['TotSales']

## 2. Model: SARIMAX

In [21]:
best_param_V1 = (1, 1, 0)
best_param_seasonal_V1 = (0, 1, 0, 12)

# best_param_V2 = (0, 2, 2)
# best_param_seasonal_V2 = (0, 2, 2, 12)

### 2.1 training

In [22]:
model = SARIMAX(y_train,
                exog=X_train, 
                order=best_param_V1,
                seasonal_order=best_param_seasonal_V1,
                enforce_stationarity=False,
                enforce_invertibility=False)
results = model.fit()


h = 12  # forecast horizon

# Training
exog_train_pred = X_train.iloc[-h:]  # 121 months
 

forecast_res = results.get_forecast(
    steps=h,
    exog=exog_train_pred              # required if you used exog
)

y_pred_forecast_mean_train = forecast_res.predicted_mean
conf_int = forecast_res.conf_int(alpha=0.05)  # 95% CI

In [23]:
y_train[-12:]

date
2025-02-01   183,989,804.00
2025-03-01   198,276,950.00
2025-04-01   204,787,548.00
2025-05-01   205,933,595.00
2025-06-01   211,838,742.00
2025-07-01   223,889,631.00
2025-08-01   159,704,179.00
2025-09-01   207,871,871.00
2025-10-01   220,481,612.00
2025-11-01   219,885,862.00
2025-12-01   222,564,373.00
2026-01-01   202,235,203.00
Name: TotSales, dtype: float64

In [24]:
y_pred_forecast_mean_train

2026-02-01   190,555,738.63
2026-03-01   204,713,916.20
2026-04-01   211,245,175.35
2026-05-01   212,387,912.36
2026-06-01   218,293,589.63
2026-07-01   230,344,393.68
2026-08-01   166,158,955.29
2026-09-01   214,326,645.11
2026-10-01   226,936,386.46
2026-11-01   226,340,636.41
2026-12-01   229,019,147.41
2027-01-01   208,689,977.41
Freq: MS, Name: predicted_mean, dtype: float64

### 2.2 testing

In [25]:
#THE OLD ONE

h = 12  # forecast horizon

forecast_res_test = results.get_forecast(
                                            steps=h,
                                            exog=X_test      # required if you used exog
                                        )

y_pred_forecast_mean_test = forecast_res_test.predicted_mean
conf_int = forecast_res_test.conf_int(alpha=0.05)  # 95% CI

In [26]:
y_original = df['TotSales'].iloc[-13] 

In [27]:
# Reversing the difference:
anchor = df['TotSales'].iloc[-13] 
final_forecast = y_pred_forecast_mean_test.cumsum() + anchor

In [28]:
final_forecast

2026-02-01     384,123,276.09
2026-03-01     590,399,695.15
2026-04-01     787,425,378.03
2026-05-01     982,135,855.18
2026-06-01   1,190,080,611.45
2026-07-01   1,403,111,658.12
2026-08-01   1,554,116,934.88
2026-09-01   1,754,925,881.97
2026-10-01   1,953,432,307.35
2026-11-01   2,160,469,091.22
2026-12-01   2,361,968,748.08
2027-01-01   2,540,005,897.34
Freq: MS, Name: predicted_mean, dtype: float64

In [29]:
y_pred_forecast_mean_test

2026-02-01   181,888,073.09
2026-03-01   206,276,419.06
2026-04-01   197,025,682.88
2026-05-01   194,710,477.16
2026-06-01   207,944,756.27
2026-07-01   213,031,046.67
2026-08-01   151,005,276.76
2026-09-01   200,808,947.08
2026-10-01   198,506,425.38
2026-11-01   207,036,783.87
2026-12-01   201,499,656.86
2027-01-01   178,037,149.26
Freq: MS, Name: predicted_mean, dtype: float64

In [39]:
df['TotSales'] = df.TotSales.fillna(y_pred_forecast_mean_test)

df['date'] = df.index

df = df[['date', 'WorkingDays', 'fixed_term', 'employment_rate', 'youth_employment_rate',
                'industry_production_prices', 'diesel_price', 'TotSales']]

In [40]:
df

,date,WorkingDays,fixed_term,employment_rate,youth_employment_rate,industry_production_prices,diesel_price,TotSales
date,,,,,,,,
2014-02-01,2014-02-01,20,"2,116,627.00",38.64,54.49,93.00,1.64,"78,038,658.00"
2014-03-01,2014-03-01,21,"2,188,435.00",38.81,55.42,92.80,1.63,"84,094,082.00"
2014-04-01,2014-04-01,20,"2,215,274.00",37.57,54.69,92.60,1.63,"86,341,669.00"
2014-05-01,2014-05-01,21,"2,255,743.00",39.78,55.44,92.50,1.63,"90,493,760.00"
2014-06-01,2014-06-01,20,"2,436,989.00",39.38,55.90,92.70,1.63,"90,851,778.00"
...,...,...,...,...,...,...,...,...
2026-09-01,2026-09-01,22,"2,714,840.53",42.35,59.79,90.85,1.62,"200,808,947.08"
2026-10-01,2026-10-01,21,"2,714,840.53",42.14,59.57,87.28,1.58,"198,506,425.38"
2026-11-01,2026-11-01,22,"2,714,840.53",42.01,59.13,86.18,1.57,"207,036,783.87"


In [41]:
df.to_excel('Italy_revenues_predicted_2026.xlsx', index=False)